# 08. Unsupervised clusters with log-ratio assignment: JECAM check and Paoay

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

In [ ]:
def unsupervised(Z, shares, method, components=3, k_range=range(2, 9), bins=3, other=False, seed=0):
    """Mahalanobis filter, standardize, PCA, then GMM (BIC) or PCIB, then log-ratio assignment."""
    from sklearn.preprocessing import StandardScaler
    Zs = StandardScaler().fit_transform(Z)
    keep, literal = clustering.mahalanobis_keep(Zs)
    pca, scores = clustering.principal_components(Zs[keep], components)
    if method == "gmm":
        model, scaler, bic = clustering.select_gmm(scores, components=k_range, seed=seed)
        labels = model.predict(scaler.transform(scores))
        mean_ari, min_ari = clustering.stability(scores, model.n_components, scaler, repeats=4)
        info = {"clusters": model.n_components, "stability_ari_mean": mean_ari, "stability_ari_min": min_ari}
    else:
        labels = clustering.isometric_bins(scores, bins=bins)
        info = {"clusters": len(set(labels))}
    sizes = pd.Series(labels).value_counts().to_dict()
    assignment, loss, near, total = clustering.logratio_assign(sizes, shares, other=other)
    full = np.full(len(Z), "removed_outlier", dtype=object)
    full[np.nonzero(keep)[0]] = [assignment[label] for label in labels]
    clusters = np.full(len(Z), -1)
    clusters[np.nonzero(keep)[0]] = labels
    return full, clusters, {**info, "kept_share": float(keep.mean()), "kept_share_literal_2_58": literal,
                            "pca_variance_explained": float(pca.explained_variance_ratio_.sum()),
                            "assignment_loss": loss, "near_optimal_assignments": near, "valid_assignments": total}

## Feature choice

In [ ]:
form = "amplitude"
names = features.OPTICAL_INDEX_NAMES
optical = features.optical_indices(arrays["s2_values"])
X, ok, columns = features.harmonic_features(optical, valid, names=names, form=form)
print("Predictors:", len(columns))

## Check on JECAM using only its aggregate crop shares

In [ ]:
run = tuning.new_run(root, "unsupervised", subset_run / "meta.json", area=AREA)
from experiments.transfer import transfer
test = np.asarray([r["partition"] == "test" for r in rows]) & ok
index = np.nonzero(test)[0]
test_rows = [rows[i] for i in index]
truth = np.asarray([r["crop"] for r in test_rows])
jecam_shares = {c: float(np.mean(truth == c)) for c in evaluation.CROPS}
print("JECAM pixel shares used as statistics:", {k: round(v, 3) for k, v in jecam_shares.items()})
records = []
for method in ("gmm", "pcib"):
    labels, _, info = unsupervised(np.nan_to_num(X[index]), jecam_shares, method)
    kept = labels != "removed_outlier"
    parcels = {}
    for row, label in zip(np.asarray(test_rows, dtype=object)[kept], labels[kept]):
        parcels.setdefault(row["partition_group"], {"crop": row["crop"], "votes": []})["votes"].append(label)
    predicted = [{"partition_group": g, "crop": v["crop"], "prediction": max(set(v["votes"]), key=v["votes"].count)}
                 for g, v in parcels.items()]
    metrics = transfer.scores(predicted)
    records.append({"method": f"{method}_logratio", "jecam_macro_f1": metrics["macro_f1"],
                    **{f"jecam_{c}_f1": metrics["per_crop"][c]["f1"] for c in evaluation.CROPS}, **info})
dev = np.asarray([r["partition"] == "train" for r in rows]) & ok
model = evaluation.fit(np.nan_to_num(X[dev]), [rows[i] for i in np.nonzero(dev)[0]])
metrics = transfer.scores(evaluation.parcel_predictions(model, np.nan_to_num(X[index]), test_rows))
records.append({"method": "supervised_european_forest", "jecam_macro_f1": metrics["macro_f1"],
                **{f"jecam_{c}_f1": metrics["per_crop"][c]["f1"] for c in evaluation.CROPS}})
tuning.save_records(run, "jecam_check", records)
display(pd.DataFrame(records).round(3))

## Paoay inputs and cropland mask

In [ ]:
paoay = json.loads((subset_run / "paoay.json").read_text())
mask = arrays["paoay_mask"]
pixels = np.argwhere(mask)
p_values = (arrays["paoay_cube"].astype(float) * .0001)[:, :, mask].transpose(2, 0, 1)
p_valid = np.isfinite(p_values).all(axis=2)
pX, pok, _ = features.harmonic_features(features.optical_indices(p_values), p_valid, names=names, form=form)
import rasterio
from crop_pipeline.analysis_workflow import resolve_stage_pointer
handoff = resolve_stage_pointer(root / "sen4agrinet_work/paoay/land_activity/land_activity_stage.json")
grid = paoay["grid"]
centres = [rasterio.Affine(*grid["transform"]) * (c + .5, r + .5) for r, c in pixels]
with rasterio.open(handoff.parent / "land_activity.tif") as raster:
    from rasterio.warp import transform as reproject_points
    xs, ys = reproject_points(grid["crs"], raster.crs, [x for x, _ in centres], [y for _, y in centres])
    land = np.asarray([value[0] for value in raster.sample(zip(xs, ys))])
from crop_pipeline.paoay import land_activity
land_names = {code: name for code, name in land_activity.LAND_CLASSES.items()}
cropland = land == 1
display(pd.Series([land_names.get(int(v), f"code {int(v)}") for v in land]).value_counts().rename("pixels").to_frame())
print("Active cropland pixels with harmonic features:", int((cropland & pok).sum()), "of", len(pixels))

## Paoay clusters assigned with Benguet crop shares

In [ ]:
statistics = pd.read_csv(root / "data/paoay/benguet_vegetable_area_2024.csv")
benguet = dict(zip(statistics.crop, statistics.area_ha / statistics.area_ha.sum()))
selection = cropland & pok
index = np.nonzero(selection)[0]
paoay_records, paoay_labels = [], {}
for method in ("gmm", "pcib"):
    for other in (False, True):
        labels, clusters, info = unsupervised(np.nan_to_num(pX[index]), benguet, method, other=other)
        key = f"{method}_{'with_other' if other else 'three_crops'}"
        paoay_labels[key] = (labels, clusters)
        shares = pd.Series(labels).value_counts(normalize=True)
        paoay_records.append({"method": key, **info, **{f"share_{c}": float(shares.get(c, 0)) for c in [*evaluation.CROPS, "other", "removed_outlier"]}})
tuning.save_records(run, "paoay_assignments", paoay_records)
display(pd.DataFrame(paoay_records).round(3))

## Reference areas

In [ ]:
from crop_pipeline.paoay import aligned
from rasterio.features import geometry_mask
from shapely.geometry import mapping
settings = json.loads((root / "settings.local.json").read_text())
_, areas, _ = aligned.study_area([Path(p) for p in paoay["kmz_paths"]], settings.get("paoay_roi_name", "Region of Interest"),
                                 list(settings.get("paoay_context_names", [])), grid["crs"])
area_rows = []
for key, (labels, _) in paoay_labels.items():
    raster = np.full(mask.shape, "not_cropland_or_no_fit", dtype=object)
    raster[tuple(pixels[index].T)] = labels
    for area in areas:
        inside = ~geometry_mask([mapping(area["geometry"])], out_shape=mask.shape, transform=rasterio.Affine(*grid["transform"])) & mask
        counts = pd.Series(raster[inside]).value_counts()
        area_rows.append({"method": key, "area": area["name"], **counts.to_dict()})
area_table = pd.DataFrame(area_rows).fillna(0)
area_table.to_csv(run / "reference_areas.csv", index=False)
display(area_table)

## Maps

In [ ]:
codes = {"carrot": 1, "cabbage": 2, "potato": 3, "other": 4, "removed_outlier": 5}
figure, axes = plt.subplots(1, len(paoay_labels), figsize=(5 * len(paoay_labels), 5))
for axis, (key, (labels, _)) in zip(axes, paoay_labels.items()):
    image = np.full(mask.shape, np.nan)
    image[tuple(pixels[index].T)] = [codes[l] for l in labels]
    axis.imshow(image, cmap="Set1", vmin=1, vmax=9, interpolation="nearest")
    axis.set_title(key)
    axis.axis("off")
show_and_save(figure, run, "paoay_logratio_maps")
print("Run folder:", run)